Book recommender -TF-IDF + Cosine similarity

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
csv_path = "https://raw.githubusercontent.com/smith8jas/DeepLearning_BookRecommender/main/data/processed/cleaned_books.csv"


In [2]:
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    df["combined_text"] = df["combined_text"].fillna("")
    return df

df = load_data(csv_path)
df.head()

,title,authors,categories,description,average_rating,ratings_count,published_year,clean_title,clean_authors,clean_categories,clean_description,popularity_score,combined_text
0,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,3.85,361.0,2004.0,gilead,marilynne robinson,fiction,a novel that readers and critics have been eag...,5.891644,gilead gilead marilynne robinson fiction ficti...
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...,3.83,5164.0,2000.0,spider s web,charles osborne agatha christie,detective and mystery stories,a new christie for christmas a full length nov...,8.549660,spider s web spider s web charles osborne agat...
2,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...,3.97,172.0,1982.0,the one tree,stephen r donaldson,american fiction,volume two of stephen donaldson s acclaimed se...,5.153292,the one tree the one tree stephen r donaldson ...
3,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",3.93,29532.0,1993.0,rage of angels,sidney sheldon,fiction,a memorable mesmerizing heroine jennifer brill...,10.293264,rage of angels rage of angels sidney sheldon f...
4,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...,4.15,33684.0,2002.0,the four loves,clive staples lewis,christian life,lewis work on the nature of love divides love ...,10.424808,the four loves the four loves clive staples le...


In [3]:
def build_tfidf(df):
    # Turn each book's combined_text into a numeric vector
    # ngram_range=(1,2): use single words and pairs of words
    vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(df["combined_text"])
    return vectorizer, matrix

vectorizer, matrix = build_tfidf(df)
print("Matrix shape:", matrix.shape)

Matrix shape: (6810, 15000)


In [4]:
def hybrid_score(text_sim, df, text_weight=0.7):
    # Combine text similarity with popularity and rating
    # score = 0.7 * text_sim + 0.3 * (0.6 * popularity + 0.4 * rating)
    pop = df["popularity_score"] / df["popularity_score"].max()
    rating = df["average_rating"] / 5.0
    meta = 0.6 * pop + 0.4 * rating
    return text_weight * text_sim + (1 - text_weight) * meta.values


In [5]:
def recommend_by_title(title, df, matrix, n=5):
    # Find the first book whose clean_title contains the query
    matches = df[df["clean_title"].str.contains(title.lower(), na=False)]
    if matches.empty:
        return pd.DataFrame()

    idx = matches.index[0]
    sim = cosine_similarity(matrix[idx], matrix).flatten()
    sim[idx] = 0  # exclude the book itself

    scores = hybrid_score(sim, df)
    top = np.argsort(scores)[::-1][:n]
    return df.loc[top, ["title", "authors", "categories", "average_rating", "published_year"]]

In [6]:
def recommend_by_query(query, df, vectorizer, matrix, n=5):
    # Vectorize the user query using the same TF-IDF model
    query_vec = vectorizer.transform([query.lower()])
    sim = cosine_similarity(query_vec, matrix).flatten()

    scores = hybrid_score(sim, df)
    top = np.argsort(scores)[::-1][:n]
    return df.loc[top, ["title", "authors", "categories", "average_rating", "published_year"]]

In [7]:
# Test 1: by title
title = input("Enter a book title: ")
print(recommend_by_title(title, df, matrix))

# Test 2: by free-text query
query = input("\nDescribe the book you want: ")
print(recommend_by_query(query, df, vectorizer, matrix))

                                                  title  \
2676  Harry Potter and the Order of the Phoenix (Boo...   
2698     Harry Potter and the Sorcerer's Stone (Book 1)   
2723    Harry Potter and the Half-Blood Prince (Book 6)   
168                       One Hundred Years of Solitude   
2710  Harry Potter and the Prisoner of Azkaban (Book 3)   

                     authors        categories  average_rating  published_year  
2676           Rowling, J.K.  Juvenile Fiction            4.49          2015.0  
2698           Rowling, J.K.  Juvenile Fiction            4.47          2015.0  
2723           Rowling, J.K.  Juvenile Fiction            4.56          2015.0  
168   Gabriel Garcia Marquez           Fiction            4.06          2003.0  
2710           Rowling, J.K.  Juvenile Fiction            4.55          2015.0  
                                     title           authors  \
4776  A Short History of Nearly Everything       Bill Bryson   
3026                Bright Ligh